In [12]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

### Adquisición y Tratamiento de Datos

In [13]:
carpeta = Path(r'..\Datos\Datos_Historicos')
dfs = {
    archivo.stem: pd.read_csv(
        archivo, sep=';', decimal=',',
        dtype={'FECHA (YYMMDD)': str, 'HORA (HHMM)': str},
    ).dropna(axis=1, how='all')
    for archivo in sorted(carpeta.glob('*.csv'))
}
for nombre, df in dfs.items():
    df['fecha_hora'] = pd.to_datetime(
        df['FECHA (YYMMDD)'] + df['HORA (HHMM)'], format='%y%m%d%H%M'
    )
    dfs[nombre] = df.drop(columns=['FECHA (YYMMDD)', 'HORA (HHMM)']).set_index('fecha_hora')

In [14]:
#La infomación presente en la página SINCA se evidencia un traslado de las estaciones dentro de la misma comuna
#Manteniendo el mismo instrumento de medición, por lo que se procede a unir las tres series de tiempo teniendo 
#estas consideraciones.
dfs["Cerrillos"] = pd.concat([dfs["Cerrillos_I"], dfs["Cerrillos"], dfs["Cerrillos_II"]]).sort_index()
dfs["Quilicura"] = pd.concat([dfs["Quilicura_I"], dfs["Quilicura"]]).sort_index() 
del [dfs["Cerrillos_I"], dfs["Cerrillos_II"], dfs["Quilicura_I"]]

In [ ]:
#Se eliminan las filas sin ningún registro, los registros preliminares y no registros no validados, a modo de trabajar solo
#con registros validados y finales por hora. 
for nombre, df in dfs.items():
    filas_antes = len(df)
    dfs[nombre] = df.drop(columns=['Registros preliminares','Registros no validados'], errors='ignore').dropna(how='all')
    print(f"{nombre}: {filas_antes - len(dfs[nombre])} filas eliminadas")

Cerrillos: 6648 filas eliminadas
Cerro_Navia: 7669 filas eliminadas
El_Bosque: 7635 filas eliminadas
Independencia: 6643 filas eliminadas
La_Florida: 12511 filas eliminadas
Las_Condes: 1915 filas eliminadas
Parque_OHiggins: 3267 filas eliminadas
Pudahuel: 4051 filas eliminadas
Puente_Alto: 6728 filas eliminadas
Quilicura: 1465 filas eliminadas
Talagante: 14389 filas eliminadas


In [4]:
def graficar(nombre, frecuencia='D'):
    if frecuencia:
        df = nombre.resample(frecuencia).mean()

    fig, ax = plt.subplots(figsize=(14, 5))
    for columna in df.columns:
        ax.plot(df.index, df[columna], linewidth=0.8, label=columna)

    ax.set_title(f"MP2.5")
    ax.set_xlabel('Fecha')
    ax.set_ylabel('MP2.5 (µg/m³)')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
